Environment Setup and Dependencies

In [ ]:
%%capture
!pip install --upgrade pip
!pip install jiwer
!pip install evaluate
!pip install tensorboard
!pip install datasets
!pip install --upgrade transformers
!pip install --upgrade torch
!pip install --upgrade torchvision
!pip install --upgrade torchaudio
!pip install librosa
!pip install numpy==2.1.0
!pip install scipy==1.11.4
!pip install librosa==0.10.1
!pip install numba==0.58.1
!pip install datasets>=2.14.0
!pip install accelerate>=0.26.0
!pip install typing_extensions --upgrade

In [ ]:
from huggingface_hub import login
login(token="HUGGING_FACE_TOKEN")

In [ ]:
!pip install --upgrade typing-extensions==4.12.2
!pip cache purge

Data Loading and Streaming Pipeline

In [ ]:
from datasets import load_dataset
from transformers import WhisperFeatureExtractor, WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Telugu", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

ds = load_dataset("kaarthu2003/SlrCvVoicesTtsDataset", streaming=True)

train_data = ds["train"].map(prepare_dataset)
test_data = ds["validation"].map(prepare_dataset)

print("Streaming pipeline ready.")

In [ ]:
!pip install torchcodec 

In [ ]:
!apt-get update && apt-get install -y ffmpeg

In [ ]:
def has_few_words(example):
    return len(example["sentence"]) < 35  
filtered_test_data = test_data.filter(has_few_words)


In [ ]:
sorted_sentences = sorted(test_data, key=lambda x: len(x["sentence"]), reverse=True)
print("30 longest sentences:")
for i, example in enumerate(sorted_sentences[:30]):
    print(f"{i+1}: ({len(example['sentence'])} chars) {example['sentence']}")

In [ ]:
print("Verifying training samples...")
for i, example in enumerate(train_data.take(2)):
    print(f"Sample {i+1} audio shape: {example['input_features'].shape}")
    print(f"Sample {i+1} label length: {len(example['labels'])}")

print("\nDataset is streaming correctly. Total size is approximately 15,811 (from metadata).")

In [ ]:
test_data = filtered_test_data

Preprocessing and Text Cleaning

In [ ]:
telugu_special_unwanted_characters = [
    'ౄ',  
    'ౢ',  
    'ౣ',  
    'ౠ',  
    'ఽ',  
    '౦', '౧', '౨', '౩', '౪', '౫', '౬', '౭', '౮', '౯',  
    'ఀ',  
    'ౘ',  
    'ౙ', 
    'ౚ', 
    '౷',  
    '‘', '’', '“', '”', '%', '.', ';', '-', ',', '/', '\\', '_', '&',  
    'G', 'P', 'S', 'e', 'l', 'n', 'r', 't', '\u200c' 
]

In [ ]:
print(f"Available columns: {train_data.column_names}")

print("\nSample row data:")
print(train_data[0])

In [ ]:
import re
chars_to_remove_regex = f'[{re.escape("".join(telugu_special_unwanted_characters))}]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"])
    return batch

In [ ]:
train_data = train_data.map(remove_special_characters)
test_data = test_data.map(remove_special_characters)

In [ ]:
repo_name = "REPO_NAME"

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="Telugu", task="transcribe")

In [ ]:
tokenizer.push_to_hub(repo_name)

In [ ]:
example = next(iter(train_data))
input_str = example["sentence"]

labels = tokenizer(input_str).input_ids
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:    {input_str}")
print(f"Decoded:  {decoded_str}")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Telugu", task="transcribe")

In [ ]:
print(train_data[0])

In [ ]:
import librosa
import re

def prepare_dataset_safe(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"])

    audio = batch["audio"]
    if audio["sampling_rate"] != 16000:
        
        audio["array"] = librosa.resample(
            audio["array"], 
            orig_sr=audio["sampling_rate"], 
            target_sr=16000
        )
        audio["sampling_rate"] = 16000

    batch["input_features"] = feature_extractor(
        audio["array"], 
        sampling_rate=16000
    ).input_features[0]
    
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

train_data = train_data.map(prepare_dataset_safe)
test_data = test_data.map(prepare_dataset_safe)

print("Resampling and cleaning pipeline is active in the stream.")

In [ ]:
print(train_data[0])

In [ ]:

!rm -rf ~/.cache/huggingface/datasets/*

!rm -rf ~/.cache/huggingface/hub/*

In [ ]:
import os
os.makedirs("/workspace/hf_cache", exist_ok=True)

os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"

In [ ]:
import librosa
import re

def prepare_dataset(batch):
    
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"])
    
    
    audio = batch["audio"]
    if audio["sampling_rate"] != 16000:
        audio["array"] = librosa.resample(
            audio["array"], 
            orig_sr=audio["sampling_rate"], 
            target_sr=16000
        )
        audio["sampling_rate"] = 16000

    
    batch["input_features"] = feature_extractor(
        audio["array"], 
        sampling_rate=16000
    ).input_features[0]
    
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
train_data = ds["train"].map(prepare_dataset)
test_data = ds["validation"].map(prepare_dataset)

print("Streaming pipeline updated. Map is now lazy and RAM-safe.")

Model Initialization

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

In [ ]:

model.generation_config.language = "telugu"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import warnings
import torch

warnings.filterwarnings("ignore", category=FutureWarning)

print(torch.cuda.memory_summary())

In [ ]:
model.config.use_cache = False

model.config.suppress_tokens = []

print(f"Update complete. Model Use Cache is now: {model.config.use_cache}")

In [ ]:

print(f"Model Use Cache: {model.config.use_cache} (Should be False)")

print(f"Forced Decoder IDs: {model.config.forced_decoder_ids}")

print(f"Tokenizer Language: {processor.tokenizer.language}")

import torch
print(f"GPU detected: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU found'}")

Fine-Tuning Setup

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

model.config.use_cache = False  

training_args = Seq2SeqTrainingArguments(
    output_dir=repo_name,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,      
    eval_accumulation_steps=1,         
    predict_with_generate=True,        
    generation_max_length=225,         
    learning_rate=1e-5,
    optim="rmsprop",
    max_steps=19760,                   
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    fp16_full_eval=True, 
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train(resume_from_checkpoint=True)

In [ ]:
!ls -d ./checkpoint-*

In [ ]:

import torch
torch.cuda.empty_cache()


In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint
last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"Checkpoint found! Resuming from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    if os.path.exists("./checkpoint-1000"):
        print("Found checkpoint-1000 in current directory. Resuming...")
        trainer.train(resume_from_checkpoint="./checkpoint-1000")
    else:
        print("No checkpoint found. Starting training from scratch.")
        trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model_id = "HUGGINGFACE_REPO_ID_HERE" 

model = WhisperForConditionalGeneration.from_pretrained(model_id)
processor = WhisperProcessor.from_pretrained(model_id)

In [ ]:
print(test_data[0])

In [ ]:
Final Performance Evaluation

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

final_wer = 100 * wer_metric.compute(predictions=predictions, references=references)
final_cer = 100 * cer_metric.compute(predictions=predictions, references=references)

print(f"Final Word Error Rate (WER): {final_wer:.2f}%")
print(f"Final Character Error Rate (CER): {final_cer:.2f}%")

In [ ]:
import torch
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() 

predictions = []
references = []

print("Generating transcriptions...")
with torch.no_grad(): 
    for i, example in enumerate(test_data):
        
        input_features = torch.tensor(example["input_features"]).unsqueeze(0).to(device)
        
        predicted_ids = model.generate(input_features)
        predicted_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        label = example["labels"]
        if isinstance(label, int):
            label = [label]
        
        label = [l for l in label if l != -100]
        reference_text = processor.batch_decode([label], skip_special_tokens=True)[0]

        predictions.append(predicted_text)
        references.append(reference_text)

# 3. Compute final percentages
final_wer = 100 * wer_metric.compute(predictions=predictions, references=references)
final_cer = 100 * cer_metric.compute(predictions=predictions, references=references)

print(f"\nWord Error Rate (WER): {final_wer:.2f}%")
print(f"Character Error Rate (CER): {final_cer:.2f}%")